# Домашнее задание №4

Материалом для практики послужит текст из [книги](https://fb2.top/pisyma-nikolay-erdman-angelina-stepanova-1928-1935-gg-457615), содержащей переписку писателя-драматурга Николая Эрдмана и его возлюбленной Ангелиной Степановой, актрисы МХАТа. 
В тексте есть сами письма (с 1928 по 1935 гг.), а также фрагменты из воспоминаний Степановой (опубликованных в 1995 году и относящихся к этому периоду).


Ваша задача — подготовить датасет писем для дальнейшего количественного анализа.

Необходимо сделать pandas датафрейм, в котором буду храниться следующие сведения (столбцы): 
- дата написания письма (желательно в формате YYYY-MM-DD), 
- автор (Степанова или Эрдман), 
- текст письма, 
- лемматизированный текст письма (без знаков препинания), 
- количество слов в письме. 

> Воспоминания сохранять в датафрейм НЕ НУЖНО!
> Сноски вида [23] и переносы строк нужно удалить из текста.

Чтобы извлечь письма и метаданные о них, нужно воспользоваться [регулярными выражениями](https://docs.python.org/3/library/re.html). 

Обратите внимание, что даты оформлены не единообразно. Продумайте все возможные варианты, перепроверьте данные. Адресант письма так же может быть указан не единообразно или не указан вовсе (но, например, есть адресат). Задача — извлечь метаданные для наибольшего количества писем.

После окончания работы над датасетом необходимо вывести df.info(), чтобы можно было посмотреть количество извлечённых дат, а также valuecounts() для столбца с адресантами.

### КРИТЕРИИ

| Что сделано | Баллы |
|---|---:|
| Извлечены тексты писем, собраны в датафрейм с указанными столбцами | 2 |
| Текст очищен и лемматизирован | 1 |
| Автор каждого письма определён: «Степанова» или «Эрдман» (включая случаи, когда автор указан не единообразно или восстанавливается через адресата) | 2 |
| Все имеющиеся даты приведены к единому формату (в оригинале даты не единообразны, например, «1928», «января 1928», «12.02.1928», «12 февраля 1928») | 3 |
| Датасет опубликован на GitHub в отдельном репозитории с файлом README.md (описание источника, структуры столбцов, методов извлечения и очистки, описаны принципы работы с отсутствующими значениями дат). Достаточно прикрепить ссылку на репозиторий. Датасет нужно опубликовать в формате .csv. | 2 |

Для поднятия настроения — басня Эрдмана:

```
СМЕТАНА.

Мы любим подмечать у недругов изъяны,
И направлять на них насмешки остриё.

Однажды молоко спросило у сметаны:
- Скажите, вы еда или питье?
Сметана молвила:
- Оставьте ваши шутки!
Действительно, я где-то в промежутке,
Но ведь важна не эта сторона
Всего важнее то, что я вкусна
И все, как правило, бывают мною сыты.

Вот так же точно и гермафродиты:
Тот, кто на свет их произвел,
Конечно, допустил ужасную небрежность.
Но ведь в конце-концов
существенен не пол,
А классовая принадлежность.
```

Читать другие басни: https://stihi.ru/diary/valery31/2014-10-11

In [2]:
from bs4 import BeautifulSoup
import re
import spacy
import pandas as pd

#парсю.

with open("457615.fb2", "r", encoding="utf-8") as f:
    textus = f.read()

soupus = BeautifulSoup(textus, "xml")
corpus = soupus.find("body")
capita = corpus.find_all("section", recursive=False)
#print(body)
#print(sections)


for f in capita: 
    titulus = f.find("title")
    if titulus is not None:
        titulus = titulus.get_text(" ", strip=True)
        
    if titulus == "1928-1935 гг.":
        epistulae_capita = f
        print(epistulae_capita)
        

    

<section>
<title>
<p>1928-1935 гг.</p>
</title>
<subtitle><strong>[Воспоминания А. И. Степановой]</strong></subtitle>
<p>Шел 1928 год. Я была замужем за Николаем Михайловичем Горчаковым, педагогом и режиссером Художественного театра. К этому времени я прошла трудный и счастливый путь артистки знаменитого театра; репетировала и играла со Станиславским, Книппер-Чеховой, Качаловым, Леонидовым, Москвиным, Тархановым, Кореневой, Халютиной. На моем счету уже значилось шесть больших ролей. Мы с мужем жили в Кривоарбатском переулке, дом наш — одну большую комнату — друзья любили за тепло и гостеприимство. К нам часто заходили коллеги-мхатовцы, писатели, художники. Всегда находилось, чем угостить их или просто накормить обедом, ужином. Частыми гостями были в доме Марков, Бабель, неразлучные тогда Олеша и Катаев, художники Дмитриев и Вильямс. Заходил и замечательный актер Театра сатиры, обаятельный Владимир Яковлевич Хенкин, остроумный, неутомимый рассказчик. Несколько раз мы принимали у себя Вс

In [3]:
#нахожу ***, запихиваю в epistulae эпистулы.

subtituli = epistulae_capita.find_all("subtitle")

separatores = []
epistulae = []

for s in subtituli:
    textus = s.get_text(" ", strip=True)

    if re.fullmatch(r"\*\s+\*\s+\*", textus):
        separatores.append(s)

# print(len(subtituli)) #81
# print(len(separatores)) #74

epistulae = []      
current = []        
is_letter = False   

for element in epistulae_capita.find_all(recursive=False):

    if element.name == "subtitle":

        if is_letter and current:
            epistulae.append(current)

        textus = element.get_text(" ", strip=True)
        is_letter = bool(re.fullmatch(r"\*\s+\*\s+\*", textus))

        current = []

    if element.name == "p" and is_letter:
        current.append(element.get_text(" ", strip=True))

if is_letter and current:
    epistulae.append(current)

print(len(epistulae))

74


In [4]:
#это ужасно
for p in epistulae:
    print(p)

['Баку,', 'Оперный театр. Гастроли М.Х.Т.,', 'А.\xa0О.\xa0Степановой.', 'Через несколько дней после вашего отъезда я заперся в своей комнате, снял пиджак и ботинки и с головой нырнул в пьесу. Только вчера я из нее вынырнул. За все это время я ни разу не вышел из дому. В отвратительной бороде я сидел за столом, поклявшись всем дорогим на свете, а значит, и вами, Лина, не вставать из-за него, пока не поставлю последней точки. Вот почему я так поздно получил ваши письма и вот почему я так долго не отвечал. Много раз я отрывался от пьесы и принимался за письмо к вам, но, не дописавши его, принимался за пьесу. Кончалось это тем, что я в совершенном отчаянье рвал и то, и другое. Простите меня, милая, я очень виноват перед вами. Бог знает, сколько грустного вы передумали, не получая моих писем. Может быть, вы даже подумали, что я позабыл вас. Если вы так подумали, то вы тоже виноваты передо мной. Сегодня, по дороге к Массу, я проходил по Кривоарбатскому переулку и смотрел в ваши окна. До чего

In [5]:
#регуярочки и функции для АТРИБУЦИИ. Я стилометрист?
def invenire_auctorem(letter):

    textus = " ".join(letter[-3:])

    signa = re.findall(r"Николай\.|Лина\.|Н\.", textus, re.IGNORECASE)

    if not signa:
        return None

    if signa[-1].lower() == "лина.":
        return "Степанова"

    return "Эрдман"

auctores = []

for letter in epistulae:
    auctor = invenire_auctorem(letter)
    auctores.append(auctor)

print("Эрдман:", auctores.count("Эрдман"))
print("Степанова:", auctores.count("Степанова"))
print("Нетути:", auctores.count(None))

print(auctores)

Эрдман: 40
Степанова: 29
Нетути: 5
['Эрдман', 'Эрдман', 'Эрдман', 'Степанова', 'Степанова', 'Эрдман', 'Степанова', 'Степанова', 'Эрдман', 'Эрдман', 'Степанова', 'Степанова', 'Степанова', 'Эрдман', 'Эрдман', None, 'Эрдман', 'Степанова', 'Эрдман', 'Эрдман', 'Эрдман', 'Эрдман', 'Эрдман', 'Степанова', 'Эрдман', 'Эрдман', 'Эрдман', None, None, 'Эрдман', 'Эрдман', 'Эрдман', 'Эрдман', 'Эрдман', 'Степанова', 'Эрдман', None, 'Эрдман', 'Степанова', 'Эрдман', 'Эрдман', 'Степанова', 'Эрдман', 'Эрдман', 'Степанова', 'Степанова', 'Эрдман', None, 'Степанова', 'Степанова', 'Степанова', 'Степанова', 'Степанова', 'Степанова', 'Степанова', 'Степанова', 'Степанова', 'Степанова', 'Степанова', 'Эрдман', 'Степанова', 'Эрдман', 'Степанова', 'Эрдман', 'Эрдман', 'Степанова', 'Эрдман', 'Эрдман', 'Эрдман', 'Степанова', 'Эрдман', 'Эрдман', 'Эрдман', 'Эрдман']


In [6]:
#никаких none'ов
for i, auctor in enumerate(auctores):
    if auctor is None:
        print("Письмо №", i + 1)
        print("\n".join(epistulae[i]))
        print("------------------------")

Письмо № 16
Москва,
проезд Художественного театра,
Художественный театр Союза ССР им.Горького,
Ангелине Осиповне Степановой.
12 декабря 33 г.
Конечно, я не утерпел и открыл посылку, как только принес домой. Я писал Тебе, что не буду ее трогать до 19-го, но это оказалось низкопробным хвастовством. Умудренный сладким опытом, я знал, что если в посылке есть папиросы, то в папиросах таится какая-нибудь неожиданность. Сознаюсь, я даже знал какая — маленькая записочка с ласковым словом в начале и с поцелуем в конце. Когда я отодрал деревянную крышку и на меня взглянула «зарвавшаяся кокетка», я еще раз понял, что Енисейск — город сюрпризов.
Целую Тебя, волшебница.
Теперь поговорим серьезно, как хозяйка с хозяйкой. Я пробовал на Тебя кричать — не помогло, попробую подействовать на Тебя лаской. Милая, хорошая, нежная, чудесная, тоненькая, замечательная, внимательная, длинноногая, трогательная, красивая, пятое-десятое — не нужно больше посылок. Ты переезжаешь, Тебе нужны деньги и гораздо больше,

In [7]:
#ручками чистим 

auctores[15] = "Эрдман"
auctores[27] = "Эрдман"
auctores[28] = "Эрдман"
auctores[36] = "Степанова"
auctores[47] = "Степанова"

print("Эрдман:", auctores.count("Эрдман"))
print("Степанова:", auctores.count("Степанова"))
print("съедены:", auctores.count(None))

Эрдман: 43
Степанова: 31
съедены: 0


In [8]:
#не ручками чищу текст. Я уборщик?

textus_mundatus = []

for letter in epistulae:
    textus = " ".join(letter)

    textus = re.sub(r"\[\d+\]", "", textus)
    textus = re.sub(r"\s+", " ", textus).strip()

    textus_mundatus.append(textus)

print("До:", epistulae[15])

print("после:", textus_mundatus[15])

До: ['Москва,', 'проезд Художественного театра,', 'Художественный театр Союза ССР им.Горького,', 'Ангелине Осиповне Степановой.', '12 декабря 33 г.', 'Конечно, я не утерпел и открыл посылку, как только принес домой. Я писал Тебе, что не буду ее трогать до 19-го, но это оказалось низкопробным хвастовством. Умудренный сладким опытом, я знал, что если в посылке есть папиросы, то в папиросах таится какая-нибудь неожиданность. Сознаюсь, я даже знал какая — маленькая записочка с ласковым словом в начале и с поцелуем в конце. Когда я отодрал деревянную крышку и на меня взглянула «зарвавшаяся кокетка», я еще раз понял, что Енисейск — город сюрпризов.', 'Целую Тебя, волшебница.', 'Теперь поговорим серьезно, как хозяйка с хозяйкой. Я пробовал на Тебя кричать — не помогло, попробую подействовать на Тебя лаской. Милая, хорошая, нежная, чудесная, тоненькая, замечательная, внимательная, длинноногая, трогательная, красивая, пятое-десятое — не нужно больше посылок. Ты переезжаешь, Тебе нужны деньги и 

In [9]:
#датирую, датирую, да никак не выдатирую

pattern1 = r"\d{1,2}\s+[а-яё]+"       
pattern2 = r"\d{1,2}[./]\w+[./-]\d+"  

dates = []

for letter in epistulae:
    data = None

    for p in letter[:8] + letter[-2:]:
        if len(p) < 50:
            if re.search(pattern1, p, re.IGNORECASE) or re.search(pattern2, p):
                data = p
                break

    dates.append(data)

for i, data in enumerate(dates):
    print(i + 1, data)

1 None
2 None
3 3 ноября 1933 г.
4 4 ноября 1933 года
5 5 ноября
6 6 ноября 1933 года
7 6 ноября
8 16 ноября
9 20 ноября 1933 г.
10 22 ноября 1933 г.
11 22 ноября
12 23 ноября
13 24 ноября
14 25 ноября 1933 г.
15 3 декабря 1933 г.
16 12 декабря 33 г.
17 12 декабря 33 г.
18 12 декабря
19 None
20 19 дек. 33 г.
21 26 дек. 33 г.
22 [1 января 34 г.]
23 10 янв. 34 г.
24 11 января
25 None
26 20.01.34 г.
27 None
28 Енисейск. 26.12 часов.
29 None
30 19 февр. 34 г.
31 None
32 None
33 18/III-34 г.
34 24 марта 34 г.
35 27 марта
36 2.04.34 г.
37 10 апреля
38 None
39 11 апреля
40 17.04.34 г.
41 (25 апреля 34 г.)
42 1 мая
43 10/V-34 г.
44 20 мая 34 года
45 1 июня
46 2 июня
47 23/VI-34 г.
48 19 августа
49 20 августа
50 21 августа
51 21 августа
52 22 августа
53 23 августа
54 23 августа
55 24 августа
56 24 августа
57 25 августа
58 25 августа
59 26 августа
60 None
61 3 сентября
62 5.09.34 г.
63 5 сентября
64 None
65 6 нояб. 34 г.
66 21 ноября
67 14.12.34 г.
68 None
69 None
70 22 февраля
71 7.03.35
72 20.

In [10]:
#даты будут арабскими, либо none

months = {
    "янв": "01", "фев": "02", "мар": "03", "апр": "04",
    "май": "05", "мая": "05", "июн": "06", "июл": "07",
    "авг": "08", "сен": "09", "окт": "10", "ноя": "11", "дек": "12"
}

roman = {
    "i": "01", "ii": "02", "iii": "03", "iv": "04",
    "v": "05", "vi": "06", "vii": "07", "viii": "08",
    "ix": "09", "x": "10", "xi": "11", "xii": "12"
}

dates_original = dates.copy()

for i, data in enumerate(dates):
    if data is None:
        continue

    data = data.lower()

    for word in re.findall(r"[а-яё]+", data):
        if word[:3] in months:
            data = data.replace(word + ".", months[word[:3]])
            data = data.replace(word, months[word[:3]])

    for r, number in roman.items():
        data = data.replace("/" + r + "-", "/" + number + "-")

    dates[i] = data

for i, data in enumerate(dates):
    print(i + 1, data)

1 None
2 None
3 3 11 1933 г.
4 4 11 1933 года
5 5 11
6 6 11 1933 года
7 6 11
8 16 11
9 20 11 1933 г.
10 22 11 1933 г.
11 22 11
12 23 11
13 24 11
14 25 11 1933 г.
15 3 12 1933 г.
16 12 12 33 г.
17 12 12 33 г.
18 12 12
19 None
20 19 12 33 г.
21 26 12 33 г.
22 [1 01 34 г.]
23 10 01 34 г.
24 11 01
25 None
26 20.01.34 г.
27 None
28 енисейск. 26.12 часов.
29 None
30 19 02 34 г.
31 None
32 None
33 18/03-34 г.
34 24 03 34 г.
35 27 03
36 2.04.34 г.
37 10 04
38 None
39 11 04
40 17.04.34 г.
41 (25 04 34 г.)
42 1 05
43 10/05-34 г.
44 20 05 34 года
45 1 06
46 2 06
47 23/06-34 г.
48 19 08
49 20 08
50 21 08
51 21 08
52 22 08
53 23 08
54 23 08
55 24 08
56 24 08
57 25 08
58 25 08
59 26 08
60 None
61 3 09
62 5.09.34 г.
63 5 09
64 None
65 6 11 34 г.
66 21 11
67 14.12.34 г.
68 None
69 None
70 22 02
71 7.03.35
72 20.03.35 г.
73 1.04.35 г.
74 None


In [11]:
#я лемматизатор.

nlp = spacy.load("ru_core_news_sm")

lemmata = []
numeri_verborum = []

for textus in textus_mundatus:
    doc = nlp(textus)
    slova = []

    for token in doc:
        if token.is_alpha:
            slova.append(token.lemma_.lower())

    lemmata.append(" ".join(slova))
    numeri_verborum.append(len(slova))

print(len(lemmata))
print(len(numeri_verborum))

74
74


In [12]:
#it's DFover

df = pd.DataFrame({
    "date_raw": dates_original,
    "author": auctores,
    "text": textus_mundatus,
    "lemmas": lemmata,
    "word_count": numeri_verborum
})

df["recipient"] = df["author"].map({
    "Эрдман": "Степанова",
    "Степанова": "Эрдман"
})

numbers = pd.Series(dates).str.findall(r"\d+")

df["day"] = pd.to_numeric(numbers.str[0], errors="coerce").astype("Int64")
df["month"] = pd.to_numeric(numbers.str[1], errors="coerce").astype("Int64")
df["year"] = pd.to_numeric(numbers.str[2], errors="coerce").astype("Int64")


display(df.head())
df.info()
print(df["author"].value_counts())

,date_raw,author,text,lemmas,word_count,recipient,day,month,year
0,NaN,Эрдман,"Баку, Оперный театр. Гастроли М.Х.Т., А. О. Ст...",баку оперный театр гастроли степанов через нес...,213,Степанова,<NA>,<NA>,<NA>
1,NaN,Эрдман,"Батум, Чаква, Цихис-Дзири, Ольге Алексеевне Ро...",батум чаква цихис дзири ольга алексеевич росто...,151,Степанова,<NA>,<NA>,<NA>
2,3 ноября 1933 г.,Эрдман,"Москва, проезд Художественного театра, Художес...",москва проезд художественный театр художествен...,122,Степанова,3,11,1933
3,4 ноября 1933 года,Степанова,"4 ноября 1933 года Дорогой, прекрасный, любимы...",ноябрь год дорогой прекрасный любимый мне хоте...,124,Эрдман,4,11,1933
4,5 ноября,Степанова,"5 ноября Дорогой мой, посылка халтурно составл...",ноябрь дорогой мой посылка халтурно составить ...,29,Эрдман,5,11,<NA>


<class 'pandas.DataFrame'>
RangeIndex: 74 entries, 0 to 73
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date_raw    60 non-null     str  
 1   author      74 non-null     str  
 2   text        74 non-null     str  
 3   lemmas      74 non-null     str  
 4   word_count  74 non-null     int64
 5   recipient   74 non-null     str  
 6   day         60 non-null     Int64
 7   month       60 non-null     Int64
 8   year        29 non-null     Int64
dtypes: Int64(3), int64(1), str(5)
memory usage: 5.5 KB
author
Эрдман       43
Степанова    31
Name: count, dtype: int64


In [13]:
df.to_csv("erdman_stepanova.csv", index=False, sep=";", encoding="utf-8-sig")